# `sessiongroups` — select, align, and combine sessions

This walks the whole workflow end-to-end. It is written to be **generic**: you
point it at *your* data folder with one variable (`DATA_DIR`) and everything else
follows. The example data shipped with this repo is one mouse's *Testing* folder,
but nothing below assumes that layout.

**The goal:** take *every* session in a folder, read + time-align all of each
session's data, then stack one data type (e.g. Nosepokes) across all sessions
into a single table you can filter and query.

```
select_sessions   ->   build_sessions   ->   combine_sessions
 (which sessions)     (read + align each)    (one structure, all sessions)
```

## 0. Configure — point at your data

Set the three paths below. `DATA_DIR` is the folder that **contains** your
session folders (they may be nested any number of levels down — see step 1).
The two YAMLs are the HARP device definitions used by the preset loaders.

> Replace these with your own paths. The defaults point at the example dataset
> in this repo, so the notebook runs as-is.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from data_conduit.datasources.monosource import RotationData
from data_conduit.sessiongroups import (
    load_session, normalise_to_zero, attach_cross_clock, build_global_clock, build_index_tables,
    DataStructureSpec,
    add_label_column,
    build_sessions,
    combine_sessions,
    default_harp_catalog,
    select_sessions,
)
import itables

if itables is not None:
    itables.init_notebook_mode()
    itables.DEFAULT_MAX_ROWS = 100
    itables
warnings.filterwarnings('ignore')   # the preset loaders print verbose hints; quieten them


def _example_root():
    """Find this repo's root (the folder with pyproject.toml) to locate the bundled
    example data. This is ONLY for the shipped example - for your own data just set
    DATA_DIR to an absolute path and you can delete this helper.
    """
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent / 'pyproject.toml').exists():
            return parent
    return Path.cwd()


_root = _example_root()

# ---- EDIT THESE to point at your own data (absolute paths are simplest) -----
DATA_DIR       = _root / 'Q_C_Analysis_Workflow' / 'data' / 'FbR_M01569522' / 'Testing'
DEVICE_YAML    = _root / 'device.yml'
SOUNDCARD_YAML = _root / 'soundcard.yml'
# -----------------------------------------------------------------------------

assert DATA_DIR.exists(), f'DATA_DIR does not exist - set it to your data folder (got: {DATA_DIR})'
print('data folder:', DATA_DIR)

## 1. Select every session in the folder

`select_sessions` walks `DATA_DIR` and returns one entry per session folder.

**Where do the sessions live?** Set `depth` to how many levels below `DATA_DIR`
the session folders are:
- `depth=0` — the session folders are the *immediate* subfolders of `DATA_DIR`;
- `depth=1` — descend one level first (here: `Day N/<session>`), etc.

In this example the `Testing` folder holds `Day N` folders, and the sessions sit
inside those, so `depth=1`. We name that intermediate level `'day'` so it is
attached to every session as metadata.

In [ ]:
sessions = select_sessions(DATA_DIR, depth=1, level_names=('day',)
                           )
print(f'found {len(sessions)} sessions under {DATA_DIR.name}/')

# Show them as a table (drop the Path object for display).
selection_table = pd.DataFrame(
    [{'session': name, **{k: v for k, v in entry.items() if k != 'path'}}
     for name, entry in sessions.items()]
)
selection_table.head(12)

### Filtering: All / Include / Exclude

By default you get **all** session folders. To narrow down, pass `include=[...]`
(keep only those named) or `exclude_names=[...]` (keep everything except those).
You can also filter the intermediate levels with `l0_selector=`, `l1_selector=`,
etc. (string, list, or a callable like `starts_with('Day 1')`).

In [ ]:
names = list(sessions)
print('ALL     :', len(select_sessions(DATA_DIR, depth=1)))
print('INCLUDE :', len(select_sessions(DATA_DIR, depth=1, include=names[:5])), '(only the 5 named)')
print('EXCLUDE :', len(select_sessions(DATA_DIR, depth=1, exclude_names=names[:5])), '(all except those 5)')
print('only Day 1 folders:', len(select_sessions(DATA_DIR, depth=1, l0_selector='Day 1 - Tactile cue')))

## 2. Choose what to extract — the data-structure catalog

A **catalog** is the list of data structures to load from each session. Each
entry knows its name and how to read it. `default_harp_catalog` gives the common
set for this lab's HARP/Bonsai rigs, which you can then toggle, add to, or trim.

In [ ]:
catalog = default_harp_catalog(DEVICE_YAML, SOUNDCARD_YAML)
print('the default structures for our use case:')
for spec in catalog:
    flags = []
    if spec.required:
        flags.append('required')
    if not spec.enabled:
        flags.append('disabled')
    print(f'   - {spec.name:18}{(" (" + ", ".join(flags) + ")") if flags else ""}')

In [ ]:
# Toggle structures on/off, drop one, or add your own. Changes are in place.
catalog.disable('video')                      # leave video out for now (kept, just off)
catalog.remove('camera')                      # drop camera frames entirely
catalog.add(DataStructureSpec(                 # add a structure not in the defaults
    name='inner_rotation',
    reader=lambda p: RotationData(experiment_directory_path=p, device_type='InnerRotation').df,
))

print('catalog now      :', catalog)
print('will be extracted:', [s.name for s in catalog.enabled_specs()])

catalog.enable('video')                        # put video back for the full demo
print('after re-enable  :', [s.name for s in catalog.enabled_specs()])

## 3. Extract & align ONE session

The alignment pipeline (`load_session` -> `normalise_to_zero` -> `build_global_clock` /
`build_index_tables`) reads every enabled structure from a session and puts them all
on one **shared, zero-based timebase**: it finds the session's earliest log,
shifts everything so that is `t = 0`, and (because we pass `global_clock`) builds
a regular clock plus a per-structure *index table* linking each structure's
samples to that clock. Structures missing from a session are simply skipped.

In [ ]:
one_path = next(iter(sessions.values()))['path']
session = load_session(one_path, catalog)
session = normalise_to_zero(session)
clock = build_global_clock(session, timestep=1.0)
index_tables = build_index_tables(session, clock)
session.metadata['global_clock'] = clock
session.metadata['index_tables'] = index_tables

print('session       :', one_path.name)
print('t0 (earliest) :', round(session.metadata['t0'], 3), 's  (subtracted so the session starts at 0)')
print('global clock  :', len(session.metadata['global_clock']), 'steps')
print('data structures extracted:')
for name in session.names:
    print('   -', name)

In [ ]:
# The events table — a DataFrame whose Time index now starts at ~0 after alignment.
session['events'].head(6)

In [ ]:
# An index table: how one structure's samples line up with the global clock.
session.metadata['index_tables']['nosepoke:Activations'].head(6)

## 4. Extract EVERY session, then combine across them

`build_sessions` runs that same extract-and-align on every selected session and
returns them as one ordered group. `combine_sessions` then stacks **one chosen
structure type** across all of them into a single object, tagged by session.

We combine all sessions in the folder here (≈30), so this cell does the real
work and may take a short while.

In [ ]:
group = build_sessions(sessions, catalog, global_clock={'timestep': 1.0})
print('group of', len(group), 'aligned sessions')
print('each provides:', group[0].names)

In [ ]:
# Stack the Nosepoke activations across ALL sessions into one DataArray.
# It stays a Nosepoke (Time x peripherals) and gains a 'label' coord naming the
# source session for every sample.
nosepokes = combine_sessions(group, dim='Time', type_name='nosepoke:Activations', label_coord='label')
nosepokes.to_dataframe()

In [ ]:
# Demo filter to only show NP_0.
# 'peripherals' (NP_0, NP_1, ...) is a DIMENSION of the DataArray, not a column,
# so we select it with xarray's .sel BEFORE converting to a DataFrame. Filtering
# columns with .filter(..., axis=1) matches nothing because the only columns are
# device/register/localID/label/Activations_data.
nosepokes.sel(peripherals='NP_0').to_dataframe()

In [ ]:
activations_clean = (
    nosepokes
    .drop_vars(["device", "register", "localID"], errors="ignore")
    .to_dataframe()
    .dropna(subset=["Activations_data"])
    .reset_index()
)

display(activations_clean)

activations_wide = (
    activations_clean
    .pivot_table(
        index=["label", "Time"],
        columns="peripherals",
        values="Activations_data",
        aggfunc="first",
    )
)

display(activations_wide)

In [ ]:
# A table of Nosepokes for ALL sessions in the folder: one row per session, with
# how many time-rows it contributed and how many nosepoke activation events it
# holds (the Activations array is sparse - NaN where a port did not fire - so we
# count the non-NaN entries across the 18 ports).
events_per_port = nosepokes.groupby('label').count()        # (session, peripheral) non-NaN counts
nosepoke_table = pd.DataFrame({
    'n_time_rows':         pd.Series(nosepokes['label'].values).value_counts(),
    'n_activation_events': events_per_port.sum('peripherals').to_series(),
}).sort_index()
nosepoke_table.index.name = 'session'
print(f'{len(nosepoke_table)} sessions combined, '
      f'{int(nosepoke_table["n_activation_events"].sum()):,} nosepoke events in total')
nosepoke_table

In [ ]:
# Any structure type combines the same way. Events -> one long table for the
# whole folder, with a 'label' column marking each row's session.
events = combine_sessions(group, dim='Time', type_name='events', label_coord='label')
print('combined events across all sessions:', len(events), 'rows')
events.head(6)

## 5. Query & tailor the combined data

The combined Nosepoke is still an `xr.DataArray`, so the `.ulookup` virtual-coord
query still works. `add_label_column` / `drop_columns` let you attach a derived
grouping (here: the day each sample came from) or drop coords when shaping a new
table.

In [ ]:
# Query: just the activations on the first Behavior board.
board0 = nosepokes.ulookup.select(device='Behavior0')
print('device=Behavior0 ->', dict(board0.sizes))

# Tag each sample with its 'day' by mapping the session label back to selection metadata.
label_to_day = {name: entry['day'] for name, entry in sessions.items()}
with_day = add_label_column(
    nosepokes, 'day', lambda da: np.array([label_to_day[s] for s in da['label'].values]), dim='Time',
)
print('added day coord ->', 'day' in with_day.coords)
print('samples per day:')
print(pd.Series(with_day['day'].values).value_counts().sort_index().head())

## 6. Cross-clock data (Neuropixels / DLC) — how it plugs in

Everything above shares the Bonsai/HARP clock. Data recorded on a *different*
clock (Neuropixels, DLC pose) is added as a structure whose spec carries a
`sync` config and whose reader returns `{'pulse_table', 'data', 'time_column'}`.
`attach_cross_clock` then fits a TTL conversion (via `data_conduit.sync.ttl`) to
bring it onto the reference clock, returning the fitted model alongside the merged
Session so the caller can cache it. These readers are **caller-supplied**, so the package never
imports `movement` or any Neuropixels library. Sketch (not run here):

```python
from data_conduit.sync.ttl import build_pulse_table, extract_ttl_segments

def npx_reader(session_path):
    raw = np.load(session_path / 'npx' / 'sync_channel.npy')
    times = np.arange(len(raw)) / 30000.0                   # 30 kHz NPX clock
    npx_pulses, _ = extract_ttl_segments(times=times, values=raw)
    spikes = pd.read_parquet(session_path / 'npx' / 'spike_times.parquet')
    return {'pulse_table': npx_pulses, 'data': spikes, 'time_column': 'spike_time'}

catalog.add(DataStructureSpec(
    name='npx',
    reader=npx_reader,
    required=False,
    sync={'reference_pulses': bonsai_pulse_table, 'use': 'start'},
))
# attach_cross_clock fits npx -> bonsai time; the returned Session gains 'npx:...' members
# and ttl_models['npx'] holds the fitted conversion.
```